# Step 6 â€” Hyperparameter Tuning

The original model used default-like hyperparameters. We search over:
- `depth`: tree depth (controls model complexity)
- `learning_rate`: step size per iteration
- `l2_leaf_reg`: L2 regularisation (prevents overfitting)

We use a **grid search evaluated on the validation set (2012â€“2014)**. Early stopping is kept active so iterations are selected automatically.

In [1]:
import pandas as pd
import numpy as np
import joblib
import itertools
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, brier_score_loss

print('All imports successful!')

All imports successful!


In [2]:
df = pd.read_csv('../data/american_bankruptcy_dataset.csv')
df['target'] = (df['status_label'] == 'failed').astype(int)

train_df = df[df['fyear'] <= 2011]
val_df   = df[(df['fyear'] >= 2012) & (df['fyear'] <= 2014)]
test_df  = df[df['fyear'] >= 2015]

FEATURES = ['X1','X2','X3','X4','X5','X6','X7','X8',
            'X9','X10','X11','X12','X13','X14','X15',
            'X16','X17','X18','Division']
cat_idx = [FEATURES.index('Division')]

train_pool = Pool(train_df[FEATURES], train_df['target'], cat_features=cat_idx)
val_pool   = Pool(val_df[FEATURES],   val_df['target'],   cat_features=cat_idx)
test_pool  = Pool(test_df[FEATURES],  test_df['target'],  cat_features=cat_idx)

y_val  = val_df['target'].values
y_test = test_df['target'].values

print('Data loaded and pools created.')

Data loaded and pools created.


## Grid Search

In [3]:
param_grid = {
    'depth':         [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'l2_leaf_reg':   [1, 5],
}

keys   = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f"Total combinations to try: {len(combos)}")

results = []

for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    m = CatBoostClassifier(
        iterations=300,
        learning_rate=params['learning_rate'],
        depth=params['depth'],
        l2_leaf_reg=params['l2_leaf_reg'],
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        early_stopping_rounds=30,
        scale_pos_weight=6,
        verbose=False
    )
    m.fit(train_pool, eval_set=val_pool)
    proba = m.predict_proba(val_pool)[:, 1]
    auc   = roc_auc_score(y_val, proba)
    f1    = f1_score(y_val, (proba >= 0.4).astype(int), pos_label=1, zero_division=0)
    results.append({**params, 'val_AUC': round(auc, 4), 'val_F1_failed': round(f1, 4),
                    'best_iteration': m.get_best_iteration()})
    d  = params['depth']
    lr = params['learning_rate']
    l2 = params['l2_leaf_reg']
    print(f"[{i+1}/{len(combos)}] depth={d} lr={lr} l2={l2} -> AUC={auc:.4f} F1={f1:.4f}")

results_df = pd.DataFrame(results).sort_values('val_AUC', ascending=False)
print()
print("=== All Results by Validation AUC ===")
print(results_df.to_string(index=False))


Total combinations to try: 12
[1/12] depth=4 lr=0.05 l2=1 -> AUC=0.8248 F1=0.2264
[2/12] depth=4 lr=0.05 l2=5 -> AUC=0.8156 F1=0.2222
[3/12] depth=4 lr=0.1 l2=1 -> AUC=0.8204 F1=0.2219
[4/12] depth=4 lr=0.1 l2=5 -> AUC=0.8224 F1=0.2316
[5/12] depth=6 lr=0.05 l2=1 -> AUC=0.8256 F1=0.2393
[6/12] depth=6 lr=0.05 l2=5 -> AUC=0.8260 F1=0.2300
[7/12] depth=6 lr=0.1 l2=1 -> AUC=0.8236 F1=0.2369
[8/12] depth=6 lr=0.1 l2=5 -> AUC=0.8198 F1=0.2238
[9/12] depth=8 lr=0.05 l2=1 -> AUC=0.8149 F1=0.2284
[10/12] depth=8 lr=0.05 l2=5 -> AUC=0.8221 F1=0.2313
[11/12] depth=8 lr=0.1 l2=1 -> AUC=0.8095 F1=0.2204
[12/12] depth=8 lr=0.1 l2=5 -> AUC=0.8227 F1=0.2342

=== All Results by Validation AUC ===
 depth  learning_rate  l2_leaf_reg  val_AUC  val_F1_failed  best_iteration
     6           0.05            5   0.8260         0.2300             258
     6           0.05            1   0.8256         0.2393             283
     4           0.05            1   0.8248         0.2264             299
     6    

## Retrain Best Model on Full Training Set and Evaluate on Test

In [4]:
best = results_df.iloc[0]
print(f'Best params: depth={best.depth}, lr={best.learning_rate}, l2={best.l2_leaf_reg}')
print(f'Best val AUC: {best.val_AUC}')

best_model = CatBoostClassifier(
    iterations=500,
    learning_rate=best['learning_rate'],
    depth=int(best['depth']),
    l2_leaf_reg=best['l2_leaf_reg'],
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    early_stopping_rounds=50,
    scale_pos_weight=6,
    verbose=False
)
best_model.fit(train_pool, eval_set=val_pool)

proba_test = best_model.predict_proba(test_pool)[:, 1]
test_auc   = roc_auc_score(y_test, proba_test)

print(f'\nTest AUC (tuned):    {test_auc:.4f}')
print(f'Test AUC (original): 0.8241')
print(f'Improvement:         {test_auc - 0.8241:+.4f}')

Best params: depth=6.0, lr=0.05, l2=5.0
Best val AUC: 0.826

Test AUC (tuned):    0.8281
Test AUC (original): 0.8241
Improvement:         +0.0040


In [5]:
from sklearn.metrics import classification_report, brier_score_loss

# Apply isotonic calibration
iso = joblib.load('../model/isotonic_calibrator.joblib')

# Refit calibrator on val probabilities from tuned model
val_proba_tuned = best_model.predict_proba(val_pool)[:, 1]
iso_tuned = joblib.load('../model/isotonic_calibrator.joblib').__class__(out_of_bounds='clip')
iso_tuned.fit(val_proba_tuned, y_val)
proba_test_cal = iso_tuned.predict(proba_test)

preds_raw = (proba_test >= 0.4).astype(int)
preds_cal = (proba_test_cal >= 0.4).astype(int)

rows = []
for label, proba, preds in [
    ('Original CatBoost (uncal.)',  None,          None),
    ('Tuned CatBoost (uncal.)',     proba_test,    preds_raw),
    ('Tuned CatBoost + Isotonic',   proba_test_cal, preds_cal),
]:
    if proba is None:
        rows.append({'Model': label, 'AUC': 0.8241, 'Brier': 0.11061,
                     'Failed Recall': 0.704, 'Failed Precision': 0.069, 'Failed F1': 0.125})
    else:
        rows.append({
            'Model':             label,
            'AUC':               round(roc_auc_score(y_test, proba), 4),
            'Brier':             round(brier_score_loss(y_test, proba), 5),
            'Failed Recall':     round(recall_score(y_test, preds, pos_label=1), 3),
            'Failed Precision':  round(precision_score(y_test, preds, pos_label=1, zero_division=0), 3),
            'Failed F1':         round(f1_score(y_test, preds, pos_label=1, zero_division=0), 3),
        })

print(pd.DataFrame(rows).set_index('Model').to_string())

                               AUC    Brier  Failed Recall  Failed Precision  Failed F1
Model                                                                                  
Original CatBoost (uncal.)  0.8241  0.11061          0.704             0.069      0.125
Tuned CatBoost (uncal.)     0.8281  0.10904          0.711             0.069      0.126
Tuned CatBoost + Isotonic   0.8275  0.02439          0.129             0.287      0.178


In [6]:
# Save tuned model and calibrator
best_model.save_model('../model/catboost_bankruptcy.cbm')
joblib.dump(iso_tuned, '../model/isotonic_calibrator.joblib')

meta = joblib.load('../model/app_metadata.joblib')
meta['tuned_params'] = {'depth': int(best['depth']),
                         'learning_rate': best['learning_rate'],
                         'l2_leaf_reg': best['l2_leaf_reg']}
meta['test_auc_tuned'] = round(test_auc, 4)
joblib.dump(meta, '../model/app_metadata.joblib')

print('Tuned model and calibrator saved.')
print(f'Best params: depth={int(best.depth)}, lr={best.learning_rate}, l2={best.l2_leaf_reg}')

Tuned model and calibrator saved.
Best params: depth=6, lr=0.05, l2=5.0
